# Double Agent: Document Assistant
Two specialized agents working together for document processing with automatic document linking.


In [28]:
# Load the OpenAI key from a .env file (see .env.example) or the environment
import os
from dotenv import load_dotenv

load_dotenv()
open_ai_api = os.environ["OPENAI_API_KEY"]
print('Credentials loaded')

Credentials loaded


In [29]:
from IPython.display import Image, display # for displaying images in Jupyter Notebooks
from openai import OpenAI
# Core imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import (
    UnstructuredPDFLoader,
    UnstructuredWordDocumentLoader,
    UnstructuredHTMLLoader,
    UnstructuredFileLoader
)
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain.tools import Tool
from langchain.tools import DuckDuckGoSearchRun
import os, re, json
from typing import Dict, Any, List, Callable
from collections import Counter
print('Imports loaded')


Imports loaded


In [30]:
# Load documents

def load_documents(directory: str):
    docs = []
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        if filename.endswith('.pdf'):
            loader = UnstructuredPDFLoader(filepath)
        elif filename.endswith('.docx'):
            loader = UnstructuredWordDocumentLoader(filepath)
        elif filename.endswith('.html') or filename.endswith('.htm'):
            loader = UnstructuredHTMLLoader(filepath)
        elif filename.endswith('.txt'):
            loader = UnstructuredFileLoader(filepath)
        else:
            continue
        docs.extend(loader.load())
    return docs

DOCS_DIR = './documents'
docs = load_documents(DOCS_DIR)
print(f'Loaded {len(docs)} documents from {DOCS_DIR}')


Loaded 9 documents from ./documents


In [31]:
# Initialize LLM and embeddings
llm = ChatOpenAI(
    openai_api_key=open_ai_api,
    model="gpt-4o",
    temperature=0.6
)

embeddings = OpenAIEmbeddings(
    openai_api_key=open_ai_api,
    model="text-embedding-3-large"
)
print('LLM and embeddings initialized')


LLM and embeddings initialized


In [32]:
# Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
split_docs = text_splitter.split_documents(docs)
print(f"Split into {len(split_docs)} chunks")


Split into 41 chunks


In [33]:
# Vector store and retrieval

db = Chroma.from_documents(split_docs, embeddings, persist_directory="./chroma_db")
print('Vector store created')

def document_retrieval(query: str):
    return db.similarity_search(query, k=2)


Vector store created


In [34]:
# OpenAI client and image generation

openai_client = OpenAI(api_key=open_ai_api)

def generate_image(description: str) -> str:
    print(f"Generating image for description: {description}")
    try:
        response = openai_client.images.generate(
            model="dall-e-3",
            prompt=description,
            size="1024x1024",
            n=1,
        )
        image_url = response.data[0].url
        return image_url
    except Exception as e:
        return f"Error generating image: {e}"

print('Image generation function ready')


Image generation function ready


In [ ]:
# Core RAG utilities

def QA(question: str) -> str:
    retriever = db.as_retriever(search_type="similarity")
    prompt_template = (
        """Answer the question based only on the following context:
    Context: {context}
    Question: {question}
    Answer:"""
    )
    prompt = ChatPromptTemplate.from_template(prompt_template)
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    answer = rag_chain.invoke({"question": question})

    related_docs = db.similarity_search(question, k=2)
    if related_docs:
        answer += "\n\n🔗 Related Documents:\n"
        for i, doc in enumerate(related_docs, 1):
            source = doc.metadata.get('source', 'Unknown')
            answer += f"{i}. {source}\n"
    return answer


def summarize_content(input_text: str) -> str:
    prompt_temp = ChatPromptTemplate.from_messages([
        ("system", "Summarize the content for general audience."),
        ("user", "{input}")
    ])
    output_parser = StrOutputParser()
    summary_chain = prompt_temp | llm | output_parser
    summary = summary_chain.invoke({"input": input_text})

    related_docs = db.similarity_search(input_text, k=2)
    if related_docs:
        summary += "\n\n🔗 Related Documents:\n"
        for i, doc in enumerate(related_docs, 1):
            source = doc.metadata.get('source', 'Unknown')
            summary += f"{i}. {source}\n"
    return summary


def link_documents(query: str) -> str:
    try:
        related_docs = db.similarity_search(query, k=4)
        if not related_docs:
            return f"No documents found related to '{query}'"
        response = f"Documents linked to '{query}':\n\n"
        for i, doc in enumerate(related_docs, 1):
            source = doc.metadata.get('source', 'Unknown')
            preview = doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content
            response += f"{i}. {source}\n   {preview}\n\n"
        return response
    except Exception as e:
        return f"Error linking documents: {e}"


In [36]:
# Translation system: glossary, utilities, agents, pipeline

GLOSSARY: Dict[str, Any] = {
    "ai": {"targets": {"default": "Artificial Intelligence", "es-ES": "Inteligencia Artificial", "fr-FR": "Intelligence Artificielle", "de-DE": "Künstliche Intelligenz"}, "strict": True},
    "hr": {"targets": {"default": "Human Resources", "es-ES": "Recursos Humanos", "fr-FR": "Ressources Humaines", "de-DE": "Personalwesen"}, "strict": False},
    "policy": {"targets": {"default": "Policy", "es-ES": "Política", "fr-FR": "Politique", "de-DE": "Richtlinie"}, "strict": True},
    "compliance": {"targets": {"default": "Compliance", "es-ES": "Cumplimiento", "fr-FR": "Conformité", "de-DE": "Compliance"}, "strict": True}
}


def advanced_llm(prompt: str, meta: Dict[str, Any]) -> str:
    try:
        response = llm.invoke(prompt)
        return response.content if hasattr(response, 'content') else str(response)
    except Exception:
        return f"[Translation LLM] {prompt.split('Text:\n')[-1].strip()}"


def _locale_target(term: str, locale: str) -> str:
    info = GLOSSARY.get(term.lower(), {})
    tgts = info.get("targets", {})
    return tgts.get(locale, tgts.get("default", ""))


def _enforce_glossary(src: str, out: str, locale: str) -> str:
    for term, info in GLOSSARY.items():
        if not info.get("strict"):
            continue
        tgt = _locale_target(term, locale)
        if not tgt:
            continue
        if re.search(rf"\b{re.escape(term)}\b", src, re.I) and not re.search(rf"\b{re.escape(tgt)}\b", out, re.I):
            out = re.sub(r"\b"+re.escape(term)+r"\b", tgt, out, count=1, flags=re.I)
    return out


def _chrF(a: str, b: str, n: int = 6, beta: float = 2.0) -> float:
    def ng(s,k): return [s[i:i+k] for i in range(max(0,len(s)-k+1))]
    A=sum([ng((a or '').lower(), i) for i in range(1,n+1)],[])
    B=sum([ng((b or '').lower(), i) for i in range(1,n+1)],[])
    ca,cb=Counter(A),Counter(B); overlap=sum((ca&cb).values())
    prec=overlap/max(1,sum(cb.values())); rec=overlap/max(1,sum(ca.values()))
    return (1+beta**2)*(prec*rec)/(beta**2*prec+rec) if (prec+rec)>0 else 0.0


def translate_with_linking(doc_id: str,
                           chunks: List[Dict[str, Any]],
                           source_lang: str,
                           target_lang: str,
                           llm_fn: Callable[[str, Dict[str, Any]], str],
                           *, style: Dict[str, Any] | None = None) -> Dict[str, Any]:
    style = style or {"tone":"neutral","formality":"standard","locale":f"{target_lang}-XX"}
    locale = style.get("locale", f"{target_lang}-XX")
    outputs, citations = [], []
    for ch in chunks:
        text = ch.get("text","")
        prompt = (
            "You are a precise corporate translator.\n"
            f"Source language: {source_lang}. Target language: {target_lang}. Style: {json.dumps(style)}.\n"
            "Return only the translation.\n"
            f"Text:\n{text}"
        )
        out = llm_fn(prompt, {"task":"translate"})
        out = _enforce_glossary(text, out, locale)
        outputs.append(out)
        citations.append({"doc_id": doc_id, "chunk_id": ch.get("id"), "tool": "translate"})
    return {"doc_id": doc_id, "text": "\n".join(outputs), "citations": citations}


def TranslationAgent(doc_id: str,
                     chunks: List[Dict[str, Any]],
                     source_lang: str,
                     target_lang: str,
                     llm_fn: Callable[[str, Dict[str, Any]], str],
                     *, style: Dict[str, Any] | None = None) -> Dict[str, Any]:
    return translate_with_linking(
        doc_id, chunks, source_lang, target_lang, llm_fn,
        style=style or {"tone":"formal","formality":"standard","locale":f"{target_lang}-XX"}
    )


def CulturalAdaptationAgent(text: str,
                            target_culture: str,
                            llm_fn: Callable[[str, Dict[str, Any]], str],
                            *, weights: Dict[str, float] | None = None,
                            overrides: Dict[str, Any] | None = None) -> Dict[str, Any]:
    weights = weights or {"tone":0.4,"local_analogy":0.3,"idiom_adaptation":0.2,"politeness":0.1}
    overrides = overrides or {}
    prompt = (
        "You are a cultural localization specialist.\n"
        f"Target culture: {target_culture}. Weights: {json.dumps(weights)}. Overrides: {json.dumps(overrides)}.\n"
        "Rewrite to fit cultural norms while preserving meaning. Return only the adapted text.\n"
        f"Text:\n{text}"
    )
    out = llm_fn(prompt, {"task":"cultural_adaptation"})
    return {"text": out, "culture": target_culture}


def QualityCheckAgent(source_text: str, translated_text: str, *, threshold: float = 0.72) -> Dict[str, Any]:
    len_ratio = len(translated_text) / max(1, len(source_text))
    chrF_score = _chrF(source_text, translated_text)
    violations = []
    for term, info in GLOSSARY.items():
        if not info.get("strict"):
            continue
        if re.search(rf"\b{re.escape(term)}\b", source_text, re.I):
            expected = [t for t in info.get("targets", {}).values() if t]
            if not any(re.search(rf"\b{re.escape(t)}\b", translated_text, re.I) for t in expected):
                violations.append({"term": term, "expected": expected})
    score = 0.3*len_ratio + 0.7*chrF_score
    passed = score >= threshold and not violations
    return {"score": round(score,3), "pass": passed, "len_ratio": round(len_ratio,3), "chrF": round(chrF_score,3), "glossary_violations": violations}


def run_translation_pipeline(doc_id: str,
                             chunks: List[Dict[str, Any]],
                             source_lang: str,
                             target_lang: str,
                             *, locale: str, doctype: str = "policy") -> Dict[str, Any]:
    res = TranslationAgent(doc_id, chunks, source_lang, target_lang, advanced_llm, 
                          style={"tone":"formal","formality":"standard","locale":locale})
    adapted = CulturalAdaptationAgent(res["text"], locale, advanced_llm)
    src_concat = "\n".join([c.get("text","") for c in chunks])
    qa = QualityCheckAgent(src_concat, adapted["text"], threshold=0.78 if doctype=="legal" else 0.72)
    return {"result": adapted, "qa": qa, "citations": res["citations"]}


def translation(input_text: str, target_language: str = "spanish") -> str:
    try:
        chunks = [{"id": "main_chunk", "text": input_text}]
        locale_map = {"spanish": "es-ES", "es": "es-ES", "french": "fr-FR", "fr": "fr-FR", "german": "de-DE", "de": "de-DE", "english": "en-US", "en": "en-US"}
        locale = locale_map.get(target_language.lower(), f"{target_language}-XX")
        result = run_translation_pipeline(
            doc_id="agent_translation",
            chunks=chunks,
            source_lang="en",
            target_lang=target_language,
            locale=locale,
            doctype="policy"
        )
        translated_text = result["result"]["text"]
        qa_info = result["qa"]
        quality_info = f"\n\n📊 Translation Quality: Score {qa_info['score']} ({'✅ PASSED' if qa_info['pass'] else '❌ NEEDS REVIEW'})"
        related_docs = db.similarity_search(input_text, k=2)
        if related_docs:
            translated_text += "\n\n🔗 Related Documents:\n"
            for i, doc in enumerate(related_docs, 1):
                source = doc.metadata.get('source', 'Unknown')
                translated_text += f"{i}. {source}\n"
        translated_text += quality_info
        if result["citations"]:
            translated_text += f"\n\n📝 Translation Citations: {len(result['citations'])} source(s) processed"
        return translated_text
    except Exception:
        prompt_temp = ChatPromptTemplate.from_messages([
            ("system", f"Translate the text into {target_language}."),
            ("user", "{input}")
        ])
        output_parser = StrOutputParser()
        translation_chain = prompt_temp | llm | output_parser
        translated_text = translation_chain.invoke({"input": input_text})
        related_docs = db.similarity_search(input_text, k=2)
        if related_docs:
            translated_text += "\n\n🔗 Related Documents:\n"
            for i, doc in enumerate(related_docs, 1):
                source = doc.metadata.get('source', 'Unknown')
                translated_text += f"{i}. {source}\n"
        return translated_text + "\n\n⚠️ Advanced translation failed, used fallback method."


In [37]:

# Tools and Agent setup (single definition)
from langchain_core.prompts import MessagesPlaceholder

search = DuckDuckGoSearchRun()
tools = [
    Tool(name="search", description="Search documents", func=document_retrieval),
    Tool(name="qa", description="Answer questions", func=QA),
    Tool(name="summarize", description="Summarize text", func=summarize_content),
    Tool(name="translate", description="Translate text to specified language. Usage: translate(text, target_language). Target languages: spanish, french, german, english. Defaults to spanish if no language specified.", func=translation),
    Tool(name="link_documents", description="Find and link related documents", func=link_documents),
    Tool(name="generate_image", description="Generate an image from a description", func=generate_image),
    search
]


adv_prompt = ChatPromptTemplate.from_messages(
    [{"role": "system", "content": """You are a prompt-writer. Your task is to produce a SINGLE, production-ready **Agent System Prompt** for an enterprise “Live Document Assistant”.
The agent must (a) retrieve from an internal document corpus,
 (b) summarize/translate/answer questions,
 (c) generate images from text prompts IF THE USER ASKED FOR GENERATING MODELS/IMAGES/LOGOS/PICTURES/ILLUSTRATIONS/DIAGRAMS use the image generation tool you MUST use the generate_image tool and return ONLY the image URL.
Do NOT answer with text instructions or suggestions for external tools.
If you cannot generate an image, say "Sorry, I can't generate images.,
 (d) ALWAYS verify freshness and fetch up-to-date external info when needed
 (e) explicitly link the internal documents used
  If a user asks about something NOT contained in the docs (or confidence is low), the agent must say so clearly, then use web search to answer and cite external sources.
"""}]
)

adv_chain = adv_prompt | llm | StrOutputParser()
resp = adv_chain.invoke({})

prompt = ChatPromptTemplate.from_messages([
    ("system", resp),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=ConversationBufferMemory(memory_key="chat_history", return_messages=True),
    verbose=True
)
print('Agent created')


Agent created


In [38]:
# Smoke tests
print("Testing Q&A:")
try:
    r1 = agent_executor.invoke({"input": "What is our companies policy on remote work?"})
    print(r1['output'][:400])
except Exception as e:
    print(f"Q&A test error: {e}")

print("\nTesting Summarization:")
try:
    r2 = agent_executor.invoke({"input": "Summarize: Company benefits include health insurance"})
    print(r2['output'][:400])
except Exception as e:
    print(f"Summarization test error: {e}")



print("\nManual link check:")
try:
    print(link_documents("employee benefits")[:400])
except Exception as e:
    print(f"Manual link error: {e}")
    
print("\nTesting Translation:")
try:
    r3 = agent_executor.invoke({"input": "Translate this to Spanish: Our company follows strict AI compliance policies for all employees."})
    print(r3['output'][:400])
except Exception as e:
    print(f"Translation test error: {e}")


Testing Q&A:


> Entering new AgentExecutor chain...

Invoking: `search` with `remote work policy`


[Document(metadata={'source': 'documents\\remote_work_policy.txt'}, page_content='COMPANYY CORPORATION\nREMOTE WORK POLICY\n\nEffective Date: January 1, 2024\n\nPURPOSE\nThis policy establishes guidelines for remote work arrangements at CompanyY Corporation, ensuring productivity, security, and effective communication while maintaining work-life balance.\n\nELIGIBILITY\nRemote work is available to employees who:\n- Have been employed for at least 6 months\n- Have satisfactory performance record\n- Have appropriate home office setup\n- Can maintain productivity remotely\n- Have manager approval\n\nWORK ARRANGEMENTS\n\nHybrid Work Model\n- Up to 3 days per week remote\n- 2 days per week in office required\n- Core collaboration days: Tuesday and Thursday\n- Flexible schedule with manager approval\n- Team meetings in person when possible\n\nFull Remote Work\n- Available for specific roles\n

In [39]:
print("Testing Q&A:")
try:
    r1 = agent_executor.invoke({"input": "What's our company policies on remote work?"})
    print(r1['output'][:400])
except Exception as e:
    print(f"Q&A test error: {e}")

Testing Q&A:


> Entering new AgentExecutor chain...

Invoking: `search` with `company remote work policy`


[Document(metadata={'source': 'documents\\remote_work_policy.txt'}, page_content='COMPANYY CORPORATION\nREMOTE WORK POLICY\n\nEffective Date: January 1, 2024\n\nPURPOSE\nThis policy establishes guidelines for remote work arrangements at CompanyY Corporation, ensuring productivity, security, and effective communication while maintaining work-life balance.\n\nELIGIBILITY\nRemote work is available to employees who:\n- Have been employed for at least 6 months\n- Have satisfactory performance record\n- Have appropriate home office setup\n- Can maintain productivity remotely\n- Have manager approval\n\nWORK ARRANGEMENTS\n\nHybrid Work Model\n- Up to 3 days per week remote\n- 2 days per week in office required\n- Core collaboration days: Tuesday and Thursday\n- Flexible schedule with manager approval\n- Team meetings in person when possible\n\nFull Remote Work\n- Available for specific

In [40]:
print("Testing Q&A:")
try:
    r1 = agent_executor.invoke({"input": "What's our vacation policy for new employees?"})
    print(r1['output'][:400])
except Exception as e:
    print(f"Q&A test error: {e}")

Testing Q&A:


> Entering new AgentExecutor chain...

Invoking: `search` with `vacation policy for new employees`


Q&A test error: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type


In [41]:
print("\nTesting Summarization:")
try:
    r2 = agent_executor.invoke({"input": "Summarize: Company benefits include health insurance"})
    print(r2['output'][:400])
except Exception as e:
    print(f"Summarization test error: {e}")


Testing Summarization:


> Entering new AgentExecutor chain...

Invoking: `summarize` with `Company benefits include health insurance`


The content suggests that the company offers health insurance as part of its employee benefits package. This means that employees have access to medical coverage, which can help with costs related to doctor visits, hospital stays, and other healthcare services. Health insurance is a common and valuable benefit that supports the well-being of employees and their families by reducing the financial burden of healthcare expenses.

🔗 Related Documents:
1. documents\benefits_guide.txt
2. documents\benefits_guide.txt
The company's benefits package includes health insurance, providing employees with medical coverage for doctor visits, hospital stays, and other healthcare services. This benefit helps reduce the financial burden of healthcare expenses, supporting the well-being of employees and their families.

> Finished chain.
The company's benefits package 

In [ ]:
print("\nTesting link documents:")
try:
    r2 = agent_executor.invoke({"input": "What documents are related to cyber security?"})
    print(r2['output'][:400])
except Exception as e:
    print(f"Summarization test error: {e}")


Testing Summarization:


> Entering new AgentExecutor chain...

Invoking: `link_documents` with `cyber security`


Documents linked to 'cyber security':

1. documents\employee_handbook.txt
   7. SAFETY AND SECURITY

Workplace Safety
- Report all accidents immediately
- Follow all safety protocols
- Emergency procedures posted throughout building
- First aid training available

Information ...

2. documents\employee_handbook.txt
   7. SAFETY AND SECURITY

Workplace Safety
- Report all accidents immediately
- Follow all safety protocols
- Emergency procedures posted throughout building
- First aid training available

Information ...

3. documents\remote_work_policy.txt
   SECURITY REQUIREMENTS

Data Security
- Use company VPN at all times
- Secure home Wi-Fi network
- Lock computer when away
- No personal devices for work
- Follow all security protocols

Confidentialit...

4. documents\remote_work_policy.txt
   SECURITY REQUIREMENTS

Data Security
- Use company VPN at all times
- Secure

In [43]:
print("\nTesting Translation:")
try:
    r3 = agent_executor.invoke({"input": "Translate this to Spanish: Our company follows strict AI compliance policies for all employees."})
    print(r3['output'][:400])
except Exception as e:
    print(f"Translation test error: {e}")


Testing Translation:


> Entering new AgentExecutor chain...

Invoking: `translate` with `Our company follows strict AI compliance policies for all employees.`


En nuestra empresa, todos los empleados deben cumplir rigurosamente con las políticas de inteligencia artificial establecidas.

🔗 Related Documents:
1. ./documents\employee_handbook.txt
2. ./documents\employee_handbook.txt


📊 Translation Quality: Score 0.728 (❌ NEEDS REVIEW)

📝 Translation Citations: 1 source(s) processedNuestra empresa aplica rigurosas políticas de cumplimiento de IA para todo el personal.

> Finished chain.
Nuestra empresa aplica rigurosas políticas de cumplimiento de IA para todo el personal.


In [44]:

print("\nTesting image generation:")
try:
    r3 = agent_executor.invoke({"input": "Generate image: Create a logo for our company."})
    print(display(Image(url=r3['output'])))
except Exception as e:
    print(f"Translation test error: {e}")


Testing image generation:


> Entering new AgentExecutor chain...

Invoking: `generate_image` with `Create a modern and sleek logo for a technology company, featuring a stylized letter 'A' with digital elements and a blue and silver color scheme.`


Generating image for description: Create a modern and sleek logo for a technology company, featuring a stylized letter 'A' with digital elements and a blue and silver color scheme.
<generated image URL removed> is a logo created for your company. You can view and download it using the link below:

![Company Logo](<generated image URL removed>)

> Finished chain.


None


In [45]:
# Fixed image generation test
print("\nTesting image generation:")
try:
    r3 = agent_executor.invoke({"input": "Generate image: Create a logo for our company."})
    image_url = r3['output']
    if image_url.startswith('http'):
        display(Image(url=image_url))
        print(f"✅ Image generated successfully: {image_url}")
    else:
        print(f"Response: {image_url}")
except Exception as e:
    print(f"Image generation test error: {e}")



Testing image generation:


> Entering new AgentExecutor chain...

Invoking: `generate_image` with `Create a modern and professional logo for a technology company, featuring abstract elements and a blue color scheme.`


Generating image for description: Create a modern and professional logo for a technology company, featuring abstract elements and a blue color scheme.
<generated image URL removed> is a logo created for your company. You can view and download it using the link below:

[View Company Logo](<generated image URL removed>)

> Finished chain.
Response: Here is a logo created for your company. You can view and download it using the link below:

[View Company Logo](<generated image URL removed>)


In [ ]:
# Optional: Real-time document synchronization (guarded)
try:
    from watchdog.observers import Observer
    from watchdog.events import FileSystemEventHandler
    import time

    class DocumentSyncHandler(FileSystemEventHandler):
        def __init__(self, vector_db, embeddings, text_splitter):
            self.vector_db = vector_db
            self.embeddings = embeddings
            self.text_splitter = text_splitter
            self.last_sync = {}
        def on_modified(self, event):
            if not event.is_directory and event.src_path.endswith(('.txt', '.pdf', '.docx', '.html')):
                self.sync_document(event.src_path)
        def on_created(self, event):
            if not event.is_directory and event.src_path.endswith(('.txt', '.pdf', '.docx', '.html')):
                self.sync_document(event.src_path)
        def sync_document(self, file_path):
            try:
                now = time.time()
                if file_path in self.last_sync and (now - self.last_sync[file_path]) < 5:
                    return
                self.last_sync[file_path] = now
                if file_path.endswith('.pdf'):
                    loader = UnstructuredPDFLoader(file_path)
                elif file_path.endswith('.docx'):
                    loader = UnstructuredWordDocumentLoader(file_path)
                elif file_path.endswith(('.html', '.htm')):
                    loader = UnstructuredHTMLLoader(file_path)
                elif file_path.endswith('.txt'):
                    loader = UnstructuredFileLoader(file_path)
                else:
                    return
                docs = loader.load()
                if not docs:
                    return
                chunks = self.text_splitter.split_documents(docs)
                self.vector_db.delete(where={"source": file_path})
                self.vector_db.add_documents(chunks)
                print(f"✅ Synced: {file_path} ({len(chunks)} chunks)")
            except Exception as e:
                print(f"❌ Sync error for {file_path}: {e}")

    sync_handler = DocumentSyncHandler(db, embeddings, text_splitter)
    observer = Observer()
    if os.path.exists(DOCS_DIR):
        observer.schedule(sync_handler, DOCS_DIR, recursive=True)
        observer.start()
        print(f"🔍 Real-time sync started for: {DOCS_DIR}")
    else:
        print(f"⚠️ Documents directory not found: {DOCS_DIR}")
except Exception as e:
    observer = None
    print(f"Real-time sync disabled: {e}")


🔍 Real-time sync started for: ./documents
